In [19]:
import yfinance as yf
import pandas as pd

# Dependent variable: NVIDIA
stock = yf.download("NVDA", start="2024-09-01", end="2026-09-01", auto_adjust=True, progress=False)

# Independent variable 1: 10-Year Treasury yield
rates = yf.download("^TNX", start="2024-09-01", end="2026-09-01", auto_adjust=True, progress=False)

# Independent variable 2: Semiconductor sector index
sox = yf.download("^SOX", start="2024-09-01", end="2026-09-01", auto_adjust=True, progress=False)

# Combine into one dataframe (align by date)
df = pd.DataFrame({
    "NVDA_Close": stock["Close"].squeeze(),
    "Rate": rates["Close"].squeeze(),
    "SOX_Close": sox["Close"].squeeze()
}).dropna()

# Build returns (dependent variable) and independent variables
df["NVDA_return"] = df["NVDA_Close"].pct_change()
df["Rate_change"] = df["Rate"].diff()          
df["SOX_return"] = df["SOX_Close"].pct_change()

# Lag the independent variables so nothing looks forward
df["Rate_change_lag1"] = df["Rate_change"].shift(1)
df["SOX_return_lag1"] = df["SOX_return"].shift(1)

model_data = df[["NVDA_return", "Rate_change_lag1", "SOX_return_lag1"]].dropna().copy()
model_data.columns = ["NVIDIA_Return", "10YTRateChange", "SOXReturn"]
model_data.to_csv("financial_dataset.csv")

In [9]:
print(model_data.shape)

(1252, 3)


In [20]:
model_data.head()

,NVIDIA_Return,10YTRateChange,SOXReturn
Date,,,
2024-09-05,0.009415,-0.076,0.002490
2024-09-06,-0.040854,-0.037,-0.005959
2024-09-09,0.035398,-0.021,-0.045167
2024-09-10,0.015309,-0.013,0.021545
2024-09-11,0.081499,-0.051,0.011866


In [11]:
model_data.tail()

,NVIDIA_Return,10YTRateChange,SOXReturn
Date,,,
2026-08-25,0.021921,-0.034,-0.027018
2026-08-26,-0.015912,-0.065,0.014433
2026-08-27,0.087380,0.025,0.002002
2026-08-28,-0.045750,0.008,0.023333
2026-08-31,0.014847,0.048,-0.034717


In [21]:
import statsmodels.api as sm

X = model_data[["10YTRateChange", "SOXReturn"]]
Y = model_data["NVIDIA_Return"]
X = sm.add_constant(X)

model = sm.OLS(Y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          NVIDIA_Return   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.6567
Date:                Sat, 05 Sep 2026   Prob (F-statistic):              0.519
Time:                        17:34:54   Log-Likelihood:                 1072.0
No. Observations:                 498   AIC:                            -2138.
Df Residuals:                     495   BIC:                            -2125.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0020      0.001      1.

In [24]:
model_data2 = df[["NVDA_return", "Rate_change", "SOX_return"]].dropna().copy()
model_data2.columns = ["NVIDIA_Return", "RateChange", "SOXReturn_unlagged"]

X2 = model_data2[["RateChange", "SOXReturn_unlagged"]]
Y2 = model_data2["NVIDIA_Return"]
X2 = sm.add_constant(X2)

model2 = sm.OLS(Y2, X2).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:          NVIDIA_Return   R-squared:                       0.516
Model:                            OLS   Adj. R-squared:                  0.514
Method:                 Least Squares   F-statistic:                     264.7
Date:                Sat, 05 Sep 2026   Prob (F-statistic):           5.90e-79
Time:                        17:38:47   Log-Likelihood:                 1255.0
No. Observations:                 499   AIC:                            -2504.
Df Residuals:                     496   BIC:                            -2491.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0002      0